In [5]:
import pandas as pd
import numpy as np
import requests

# -----------------------------------
# 1. 필터
# -----------------------------------
def filter_station(df, station_id):
    mask = (
        df['시작_대여소_ID'].astype(str).str.contains(station_id, na=False) |
        df['종료_대여소_ID'].astype(str).str.contains(station_id, na=False)
    )
    return df.loc[mask].copy()


# -----------------------------------
# 2. 도착시간 보정
# -----------------------------------
def update_end_time(df, station_id):
    df = df.copy()

    df["전체_이용_분"] = pd.to_timedelta(df["전체_이용_분"], errors="coerce")
    df["기준_날짜"] = pd.to_datetime(df["기준_날짜"])

    mask = df["종료_대여소_ID"] == station_id

    hours = df.loc[mask, "전체_이용_분"].dt.total_seconds() / 3600.0
    raw = np.floor(df.loc[mask, "시간대"] - hours).astype(int)

    day_shift = np.floor_divide(raw, 24)
    adj_hour = raw - day_shift * 24

    df.loc[mask, "기준_날짜"] += pd.to_timedelta(day_shift, unit="D")
    df.loc[mask, "시간대"] = adj_hour
    df.loc[mask, "집계_기준"] = "도착시간"

    return df.sort_values(["기준_날짜", "시간대"]).reset_index(drop=True)


# -----------------------------------
# 3. 시간 확장 (🔥 datetime 유지)
# -----------------------------------
def build_hourly(df):
    df = df.copy()

    df["기준_날짜"] = pd.to_datetime(df["기준_날짜"])

    start = df["기준_날짜"].min()
    end = df["기준_날짜"].max()

    hours = pd.date_range(start, end + pd.Timedelta(hours=23), freq="h")

    base = pd.DataFrame({"_dt": hours})

    base = pd.concat([
        base.assign(집계_기준="출발시간"),
        base.assign(집계_기준="도착시간")
    ])

    # ✅ 핵심: datetime 유지
    base["기준_날짜"] = base["_dt"].dt.floor("D")
    base["시간대"] = base["_dt"].dt.hour
    base = base.drop(columns="_dt")

    counts = (
        df.groupby(["기준_날짜", "시간대", "집계_기준"])["전체_건수"]
        .sum()
        .reset_index()
    )

    base = base.merge(counts, on=["기준_날짜", "시간대", "집계_기준"], how="left")
    base["전체_건수"] = base["전체_건수"].fillna(0)

    base.loc[base["집계_기준"] == "출발시간", "전체_건수"] *= -1

    return base.sort_values(["기준_날짜", "시간대"]).reset_index(drop=True)


# -----------------------------------
# 4. 날씨 (한 번만)
# -----------------------------------
def get_weather(start_date, end_date, lat=37.6, lon=126.93):
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,snowfall",
        "timezone": "Asia/Seoul",
    }

    data = requests.get(url, params=params).json()["hourly"]

    weather = pd.DataFrame({
        "time": pd.to_datetime(data["time"]),
        "온도": data["temperature_2m"],
        "습도": data["relative_humidity_2m"],
        "강수량": data["precipitation"],
        "적설량": data["snowfall"],
    })

    # ✅ datetime 유지
    weather["기준_날짜"] = weather["time"].dt.floor("D")
    weather["시간대"] = weather["time"].dt.hour

    return weather.drop(columns="time")


def merge_weather(df, weather):
    df = df.copy()
    df["기준_날짜"] = pd.to_datetime(df["기준_날짜"])
    weather["기준_날짜"] = pd.to_datetime(weather["기준_날짜"])

    return df.merge(weather, on=["기준_날짜", "시간대"], how="left")


# -----------------------------------
# 5. 시간 피처
# -----------------------------------
def add_time_features(df):
    df = df.copy()

    date = pd.to_datetime(df["기준_날짜"])

    df["dow"] = date.dt.dayofweek
    df["day_type"] = (df["dow"] >= 5).astype(int)

    df["hour_sin"] = np.sin(2*np.pi*df["시간대"]/24)
    df["hour_cos"] = np.cos(2*np.pi*df["시간대"]/24)

    df["rush_hour"] = (
        ((df["시간대"].between(7,9)) | (df["시간대"].between(17,19)))
    ).astype(int)

    return df


# -----------------------------------
# 6. lag
# -----------------------------------
def add_lag_features(df):
    df = df.sort_values(["기준_날짜", "시간대"]).copy()

    df["lag24"] = df["전체_건수"].shift(24)
    df["lag168"] = df["전체_건수"].shift(168)
    df["rolling24"] = df["전체_건수"].shift(1).rolling(24).mean()

    return df


# -----------------------------------
# 7. 불쾌지수
# -----------------------------------
def add_discomfort(df):
    df = df.copy()
    df["불쾌지수"] = (
        0.81 * df["온도"]
        + 0.01 * df["습도"] * (0.99 * df["온도"] - 14.3)
        + 46.3
    )
    return df


# -----------------------------------
# 8. 전체 파이프라인
# -----------------------------------
def make_dataset(df, station_id, weather):
    df = filter_station(df, station_id)
    df = update_end_time(df, station_id)
    df = build_hourly(df)
    df = merge_weather(df, weather)
    df = add_time_features(df)
    df = add_lag_features(df)
    df = add_discomfort(df)
    return df


# -----------------------------------
# 9. 실행 코드
# -----------------------------------
from lightgbm import LGBMRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# 데이터 로드
station = pd.read_parquet("../../Data/sort_data/2024_data.parquet")
station2025 = pd.read_parquet("../../Data/sort_data/2025_data.parquet")

# 날씨 (1번만)
weather = get_weather("2024-01-01", "2025-12-31")

# 데이터 생성
train_df = make_dataset(station, "ST-2247", weather)
test_df = make_dataset(station2025, "ST-2247", weather)

# feature
feature_cols = [
    "시간대", "day_type",
    "온도", "습도", "불쾌지수",
    "강수량", "적설량",
    "lag24", "lag168",
    "hour_sin", "hour_cos",
    "rolling24",
    "rush_hour"
]

target_col = "전체_건수"

train_df = train_df.dropna(subset=feature_cols + [target_col])
test_df = test_df.fillna(method="ffill")

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

# 모델
model = LGBMRegressor(random_state=42)

param_grid = {
    "n_estimators": [300],
    "learning_rate": [0.05],
    "num_leaves": [31, 63],
}

tscv = TimeSeriesSplit(n_splits=3)

search = GridSearchCV(model, param_grid, cv=tscv, scoring="r2")
search.fit(X_train, y_train)

best_model = search.best_estimator_

# 평가
def score(y, pred):
    return {
        "R2": r2_score(y, pred),
        "MAE": mean_absolute_error(y, pred),
        "RMSE": np.sqrt(mean_squared_error(y, pred))
    }

print("TRAIN:", score(y_train, best_model.predict(X_train)))
print("CV R2:", search.best_score_)
print("TEST :", score(y_test, best_model.predict(X_test)))
print("BEST :", search.best_params_)

/var/folders/yv/4nqj4qhx05s3jq59y10qjzyc0000gn/T/ipykernel_19893/4009074662.py:34: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[13 18 17 ... 12 13  6]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[mask, "시간대"] = adj_hour
/var/folders/yv/4nqj4qhx05s3jq59y10qjzyc0000gn/T/ipykernel_19893/4009074662.py:34: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[12  7 20 ... 13 23  6]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[mask, "시간대"] = adj_hour
/var/folders/yv/4nqj4qhx05s3jq59y10qjzyc0000gn/T/ipykernel_19893/4009074662.py:212: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  test_df = test_df.fillna(method="ffill")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000720 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 761
[LightGBM] [Info] Number of data points in the train set: 4350, number of used features: 13
[LightGBM] [Info] Start training from score -0.133333
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000549 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 852
[LightGBM] [Info] Number of data points in the train set: 8700, number of used features: 13
[LightGBM] [Info] Start training from score -0.204598
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000586 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough,